In [ ]:
import pandas as pd
import numpy as np
!pip install pandas sqlalchemy pymysql openpyxl

In [ ]:
ID_COLS = {"article_id": str, "product_code": str}

In [ ]:
articles = pd.read_csv('../data/bronze/hm/articles_sample.csv', dtype=ID_COLS)
customers = pd.read_csv('../data/bronze/hm/customers_sample.csv', dtype=ID_COLS)
transactions = pd.read_csv('../data/bronze/hm/transactions_sample.csv', dtype=ID_COLS)

In [ ]:
flat = pd.read_csv('../data/silver/hm_flat.csv', dtype=ID_COLS, parse_dates=['t_dat'])

In [ ]:
print(customers.head(5))


In [ ]:
print(articles.head(5))

In [ ]:
print(transactions.head(5))

In [ ]:
# 1. Data Anomalies
# Insertion anomalies
unsold = articles[~articles['article_id'].isin(transactions['article_id'])]
print(unsold.head(1))

In [ ]:
# deletion anolamies
del_summary = flat.groupby('customer_id')['customer_id'].count().reset_index(name='count')
print(del_summary[del_summary['count'] == 1].head(1)[['customer_id']])

In [ ]:
# update anomalies
update_summary = flat.groupby('article_id')['article_id'].count().reset_index(name='count')
print(update_summary[update_summary['count'] > 30].head(1))

In [ ]:
# 2. Normalization
# 1NF
KEY = ["customer_id", "t_dat", "sales_channel_id", "article_id", "price"]
other = [c for c in flat.columns if c not in KEY]

nf1 = (flat.groupby(KEY + other, dropna=False)
           .size().reset_index(name="quantity"))

assert not nf1.duplicated(subset=KEY).any()
print(len(flat), "→", len(nf1), "rows")
print("purchases split by price:", (nf1.groupby(KEY[:-1]).size() > 1).sum())

In [ ]:
# 2NF
CUST_COLS = ["customer_id", "age", "club_member_status", "fashion_news_frequency", "FN", "Active", "postal_code"]
ART_COLS = ["article_id", "product_code", "prod_name", "product_type_no", "product_type_name", "product_group_name",
            "graphical_appearance_no", "graphical_appearance_name", "colour_group_code", "colour_group_name",
            "department_no", "department_name", "section_no", "section_name",
            "index_code", "index_name", "index_group_no", "index_group_name"]
ORDER_KEY = ["customer_id", "t_dat", "sales_channel_id"]

# Partial dependencies → their own tables
customers = nf1[CUST_COLS].drop_duplicates()
articles = nf1[ART_COLS].drop_duplicates()
assert customers.customer_id.is_unique and articles.article_id.is_unique

# Orders: one row per customer + date + channel, with a new ID
orders = nf1[ORDER_KEY].drop_duplicates().sort_values(["t_dat", "customer_id"]).reset_index(drop=True)
orders.insert(0, "order_id", range(1, len(orders) + 1))

# Order items: what's left, linked to orders by order_id
order_items = nf1.merge(orders, on=ORDER_KEY)[["order_id", "article_id", "price", "quantity"]]
order_items.insert(0, "order_item_id", range(1, len(order_items) + 1))

# Checks: nothing lost, no duplicates
assert len(order_items) == len(nf1)
assert not order_items.duplicated(["order_id", "article_id", "price"]).any()
for name, df in [("customers", customers), ("articles", articles), ("orders", orders), ("order_items", order_items)]:
    print(f"{name}: {len(df):,} rows")

In [ ]:
# 3NF
def holds(df, x, y):
    return (df.groupby(x)[y].nunique() <= 1).all()

def normalize_articles(art):
    # Verify each dependency before splitting on it
    checks = [("product_type_no", ["product_type_name", "product_group_name"]),
              ("graphical_appearance_no", ["graphical_appearance_name"]),
              ("colour_group_code", ["colour_group_name"]),
              ("department_no", ["department_name", "section_no", "index_code"]),
              ("section_no", ["section_name"]),
              ("index_code", ["index_name", "index_group_no"]),
              ("index_group_no", ["index_group_name"])]
    for x, ys in checks:
        for y in ys:
            assert holds(art, x, y), f"{x} -> {y} does NOT hold"

    product_types = art[["product_type_no", "product_type_name", "product_group_name"]].drop_duplicates()
    graphical_appearances = art[["graphical_appearance_no", "graphical_appearance_name"]].drop_duplicates()
    colour_groups = art[["colour_group_code", "colour_group_name"]].drop_duplicates()
    sections = art[["section_no", "section_name"]].drop_duplicates()

    # Recursive category table: index groups (top level) and indexes (point to their group)
    groups = art[["index_group_no", "index_group_name"]].drop_duplicates()
    groups = pd.DataFrame({"category_id": "G" + groups.index_group_no.astype(str),
                           "category_name": groups.index_group_name,
                           "category_level": "index_group", "parent_category_id": None})
    idx = art[["index_code", "index_name", "index_group_no"]].drop_duplicates()
    idx = pd.DataFrame({"category_id": "I" + idx.index_code,
                        "category_name": idx.index_name,
                        "category_level": "index",
                        "parent_category_id": "G" + idx.index_group_no.astype(str)})
    categories = pd.concat([groups, idx], ignore_index=True)

    departments = art[["department_no", "department_name", "section_no", "index_code"]].drop_duplicates()
    departments["category_id"] = "I" + departments.pop("index_code")

    articles_3nf = art[["article_id", "product_code", "prod_name", "product_type_no",
                        "graphical_appearance_no", "colour_group_code", "department_no"]]
    return dict(articles=articles_3nf, product_types=product_types, graphical_appearances=graphical_appearances,
                colour_groups=colour_groups, departments=departments, sections=sections, categories=categories)

# Build from the FULL article sample so the 1,000 unsold articles are included
tables = normalize_articles(pd.read_csv('../intermediate/articles_sample.csv', dtype=ID_COLS))

tables["customers"] = customers
tables["orders"] = orders
tables["order_items"] = order_items
tables["sales_channels"] = pd.DataFrame({"sales_channel_id": [1, 2], "channel_name": ["Store", "Online"]})

print("fashion_news_frequency -> FN holds?", holds(customers, "fashion_news_frequency", "FN"))
for name, df in tables.items():
    print(f"{name}: {len(df):,} rows")

In [ ]:
from openpyxl.styles import Font

# Pick 3 customers to trace: one with a multi-unit purchase + two small baskets
sizes = flat.groupby("customer_id").size()
pick = (nf1.loc[nf1.quantity > 1, "customer_id"].loc[lambda s: s.map(sizes) <= 8].unique()[:1].tolist()
        + sizes[sizes.between(2, 6)].sample(2, random_state=42).index.tolist())

# Filter every table down to those customers and the articles they bought
o = orders[orders.customer_id.isin(pick)]
oi = order_items[order_items.order_id.isin(o.order_id)]
c = customers[customers.customer_id.isin(pick)]
t = tables
a3 = t["articles"][t["articles"].article_id.isin(oi.article_id)]
d = t["departments"][t["departments"].department_no.isin(a3.department_no)]
cat = t["categories"]
used = cat[cat.category_id.isin(d.category_id)]
cat = cat[cat.category_id.isin(used.category_id) | cat.category_id.isin(used.parent_category_id)]

sheets = {
    "1NF": [
        ("Flat table BEFORE 1NF (duplicate rows = multi-unit purchases)", flat[flat.customer_id.isin(pick)]),
        ("1NF | PK: (customer_id, t_dat, sales_channel_id, article_id, price) | duplicates collapsed into quantity",
         nf1[nf1.customer_id.isin(pick)]),
    ],
    "2NF": [
        ("customers | PK: customer_id | customer_id → age, club_member_status, fashion_news_frequency, FN, Active, postal_code", c),
        ("orders | PK: order_id | FK: customer_id", o),
        ("order_items | PK: order_item_id | FK: order_id, article_id | junction table (orders ↔ articles, M:N)", oi),
        ("articles | PK: article_id | article_id → all article & category attributes", articles[articles.article_id.isin(oi.article_id)]),
    ],
    "3NF": [
        ("customers | PK: customer_id | unchanged from 2NF", c),
        ("sales_channels | PK: sales_channel_id", t["sales_channels"]),
        ("orders | PK: order_id | FK: customer_id, sales_channel_id", o),
        ("order_items | PK: order_item_id | FK: order_id, article_id", oi),
        ("articles | PK: article_id | FK: product_type_no, graphical_appearance_no, colour_group_code, department_no", a3),
        ("product_types | PK: product_type_no", t["product_types"][t["product_types"].product_type_no.isin(a3.product_type_no)]),
        ("graphical_appearances | PK: graphical_appearance_no",
         t["graphical_appearances"][t["graphical_appearances"].graphical_appearance_no.isin(a3.graphical_appearance_no)]),
        ("colour_groups | PK: colour_group_code", t["colour_groups"][t["colour_groups"].colour_group_code.isin(a3.colour_group_code)]),
        ("departments | PK: department_no | FK: section_no, category_id", d),
        ("sections | PK: section_no", t["sections"][t["sections"].section_no.isin(d.section_no)]),
        ("categories | PK: category_id | FK: parent_category_id → categories (recursive)", cat),
    ],
}

import os
os.makedirs("../data/output", exist_ok=True)

with pd.ExcelWriter("../data/output/normalization.xlsx", engine="openpyxl") as xw:
    for sheet, blocks in sheets.items():
        row = 0
        for title, df in blocks:
            df.to_excel(xw, sheet_name=sheet, startrow=row + 1, index=False)
            xw.sheets[sheet].cell(row=row + 1, column=1, value=title).font = Font(bold=True, size=12)
            row += len(df) + 4

print("Saved ../data/output/normalization.xlsx | traced customers:", len(pick))

In [ ]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from getpass import getpass

host = input("RDS endpoint: ")
user = input("Master username: ")
pw = getpass("Password: ")
engine = create_engine(f"mysql+pymysql://{user}:{quote_plus(pw)}@{host}:3306/secondlook")

# Match column names and types to schema.sql
t = {k: v.copy() for k, v in tables.items()}
t["orders"] = t["orders"].rename(columns={"t_dat": "order_date"})
t["orders"]["order_date"] = pd.to_datetime(t["orders"]["order_date"]).dt.date
t["customers"] = t["customers"].rename(columns={"FN": "fn", "Active": "active"})
t["customers"][["fn", "active"]] = t["customers"][["fn", "active"]].astype(float).fillna(0).astype(int)
t["customers"]["age"] = pd.to_numeric(t["customers"]["age"]).astype("Int64")
t["order_items"]["price"] = t["order_items"]["price"].astype(float).round(16)
assert not t["order_items"].duplicated(["order_id", "article_id", "price"]).any()

# Parent tables before child tables, so every foreign key already exists
LOAD_ORDER = ["product_types", "graphical_appearances", "colour_groups", "sections", "categories",
              "departments", "sales_channels", "customers", "articles", "orders", "order_items"]

for name in LOAD_ORDER:
    t[name].to_sql(name, engine, if_exists="append", index=False, chunksize=1000, method="multi")
    print(f"loaded {name}: {len(t[name]):,} rows")

# Verify row counts in the database
with engine.connect() as conn:
    for name in LOAD_ORDER:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {name}")).scalar()
        print(f"{name}: {n:,} rows in RDS")